# Training Through Phase Retrieval

A magnitude-domain model cannot be trained on a waveform loss unless the step that turns
magnitudes back into a waveform carries a gradient. This notebook establishes, by
measurement rather than assertion, **which parts of the `cool_frames` torch backend are
differentiable**, gradchecks the one that matters, and optimises through it.

It then reports an inconvenient result honestly: a correct gradient is not by itself a
reason to use a loss. Section 4 measures why, and what to do instead. That is worth knowing
before building a pipeline around one.

> **Note.** Earlier revisions of this notebook used *Diff-RTPGHI* — the differentiable
> fixed-order PGHI variant from C. Hollomey, *Differentiable Real-Time Phase Reconstruction
> for Non-Uniform Filterbanks*. That algorithm is not part of the toolbox; it lives with the
> paper's own code. What the toolbox ships is `cool_frames.torch.phase.gla`, a native torch
> port of (fast) Griffin-Lim, and that is what is used here.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'cool-frames @ git+https://github.com/allthatsounds/cool-frames.git'],
                   check=True)

import warnings

import matplotlib.pyplot as plt

import numpy as np
import torch

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100

torch.manual_seed(0)
DT, CDT = torch.float64, torch.complex128
print('torch', torch.__version__)

In [ ]:
from cool_frames.torch.filterbanks import filterbank, filterbankdual, ifilterbank
from cool_frames.torch.filters import audfilters
from cool_frames.torch.phase import filterbankconstphase, gla

print('All imports OK.')

## 1. Which paths carry a gradient?

The torch backend is not uniformly differentiable, and the exceptions are not obvious from
the names. Rather than take the docstrings on trust, the next cell calls `.backward()`
through each path and reports what actually arrives.

In [ ]:
fs, Ls = 8000, 2000
with warnings.catch_warnings():
    warnings.simplefilter('error')      # a non-frame geometry must not slip past
    g, a, fc, L, info = audfilters(fs, Ls)
M = len(g)
print(f'Filterbank: M={M} channels, L={L}, '
      f"admissible={info['admissible']['is_frame']}")

x = torch.randn(Ls, dtype=DT)
xp = torch.zeros(L, dtype=DT)
xp[:Ls] = x
c = filterbank(xp, g, a, L=L)
n_coef = [cm.numel() for cm in c]
TOT = sum(n_coef)
print(f'{TOT} coefficients total ({sum(n_coef)/L:.1f}x redundant)')

gd = filterbankdual(g, a, L)


def report(label, fn):
    mags = [cm.abs().detach().clone().requires_grad_(True) for cm in c]
    try:
        out = fn(mags)
        loss = (out ** 2).sum()
        loss.backward()
        tot = sum(float(m.grad.abs().sum()) for m in mags if m.grad is not None)
        finite = all(bool(torch.isfinite(m.grad).all())
                     for m in mags if m.grad is not None)
        status = f'gradient reaches magnitudes: sum|d| = {tot:.3f}, finite={finite}'
    except Exception as exc:                       # report, don't hide
        status = f'NO GRADIENT — {type(exc).__name__}: {str(exc)[:70]}'
    print(f'{label:<44s} {status}')


# a) analysis -> synthesis round trip
report('ifilterbank(magnitudes as coefficients)',
       lambda mags: ifilterbank([m.to(CDT) for m in mags], gd, a, Ls))

# b) magnitudes x FIXED phase -> synthesis
with torch.no_grad():
    c_pghi, _mask = filterbankconstphase(xp, g, a, L=L, fc=fc, fs=fs)
    frozen_phase = [torch.angle(p) for p in c_pghi]
report('magnitudes x frozen PGHI phase -> synth',
       lambda mags: ifilterbank(
           [m.to(CDT) * torch.exp(1j * p) for m, p in zip(mags, frozen_phase)],
           gd, a, Ls))

# c) torch gla: magnitudes -> phase -> waveform, in one call
report('gla (magnitudes -> waveform)',
       lambda mags: gla(mags, g, a, L=L, Ls=Ls, real=True,
                        maxit=4, method='fgla', startphase='zero')[1])

# d) PGHI itself
report('filterbankconstphase (PGHI)',
       lambda mags: torch.stack([p.abs().sum() for p in
                                 filterbankconstphase(xp, g, a, L=L, fc=fc, fs=fs)[0]]))

So: analysis and synthesis are differentiable, `gla` is differentiable, and PGHI is not —
its heap traversal order is a discrete function of the magnitudes, so there is nothing to
differentiate. PGHI can still be used inside a graph by *freezing* its output as a constant
phase, which is the second row above.

(`ifilterbankiter` is the other exception: it detaches internally. Use `ifilterbank`.)

## 2. Is the gradient *correct*?

Non-zero is not the same as right. The check below compares the analytic gradient of `gla`
against a central finite difference on a handful of coefficients.

In [ ]:
def gla_loss(mags, maxit=4):
    y = gla(mags, g, a, L=L, Ls=Ls, real=True,
            maxit=maxit, method='fgla', startphase='zero')[1]
    return (y ** 2).sum()


mags = [cm.abs().detach().clone().requires_grad_(True) for cm in c]
gla_loss(mags).backward()

ch, eps = 3, 1e-6
print(f'{"idx":>5s} {"analytic":>18s} {"finite diff":>18s} {"rel. err":>11s}')
for idx in (0, 5, 17):
    base = [m.detach().clone() for m in mags]

    plus = [t.clone() for t in base]
    plus[ch][idx] += eps
    minus = [t.clone() for t in base]
    minus[ch][idx] -= eps
    with torch.no_grad():
        fd = (gla_loss(plus) - gla_loss(minus)) / (2 * eps)

    an = float(mags[ch].grad[idx])
    rel = abs(an - float(fd)) / max(abs(an), 1e-30)
    print(f'{idx:5d} {an:18.8e} {float(fd):18.8e} {rel:11.1e}')

The analytic and numerical gradients agree to several digits, so the chain really is intact
— not merely non-zero.

## 3. Optimising through the phase step

The point of a differentiable phase step is that a loss defined on the **waveform** can
reach back and change the **magnitudes**. The cleanest demonstration is to do exactly that
with no network in the way: take the magnitudes of a noisy signal as free parameters and
descend on the waveform error against the clean signal.

If the gradient is usable — not merely present — the loss falls monotonically and the
reconstruction improves. Nothing here is a denoising method; it is an instrument reading.

In [ ]:
def analyse(sig_np):
    xp = torch.zeros(L, dtype=DT)
    xp[:Ls] = torch.as_tensor(np.asarray(sig_np)[:Ls], dtype=DT)
    return torch.cat([cm.abs().reshape(-1) for cm in filterbank(xp, g, a, L=L)])


def unflatten(flat):
    out, o = [], 0
    for nm in n_coef:
        out.append(flat[o:o + nm])
        o += nm
    return out


def recon(mag_flat, maxit=6):
    return gla(unflatten(mag_flat), g, a, L=L, Ls=Ls, real=True,
               maxit=maxit, method='fgla', startphase='zero')[1]


def si_sdr(ref, est):
    # Scale-invariant SDR: project onto the reference first, so a global gain
    # error is not counted as distortion.
    n = min(len(ref), len(est))
    r, e = ref[:n], est[:n]
    alpha = (e * r).sum() / ((r * r).sum() + 1e-30)
    tgt = alpha * r
    return 10 * torch.log10((tgt ** 2).sum() / (((e - tgt) ** 2).sum() + 1e-30))

In [ ]:
# A clean signal and a 5 dB noisy version of it.
t = np.arange(Ls) / fs
clean = (np.sin(2*np.pi*300*t) + 0.6*np.sin(2*np.pi*1100*t)
         + 0.3*np.sin(2*np.pi*2300*t))
clean = clean / np.max(np.abs(clean))
rng = np.random.default_rng(0)
noise = rng.standard_normal(Ls)
noise *= np.sqrt(np.sum(clean**2) / 10**(5/10) / np.sum(noise**2))
noisy = clean + noise

clean_w = torch.as_tensor(clean, dtype=DT)
mag = analyse(noisy).clone().requires_grad_(True)
opt = torch.optim.Adam([mag], lr=1e-2)

hist = []
for it in range(60):
    opt.zero_grad()
    loss = ((recon(mag) - clean_w) ** 2).mean()
    loss.backward()
    opt.step()
    with torch.no_grad():
        mag.clamp_(min=0.0)              # magnitudes are non-negative
        hist.append((float(loss), float(si_sdr(clean_w, recon(mag)))))
    if (it + 1) % 10 == 0:
        print(f'  iter {it+1:3d}: loss={hist[-1][0]:.5f}  SI-SDR={hist[-1][1]:6.2f} dB')

print()
print(f'loss   {hist[0][0]:.5f} -> {hist[-1][0]:.5f}')
print(f'SI-SDR {hist[0][1]:.2f} dB -> {hist[-1][1]:.2f} dB '
      f'({hist[-1][1] - hist[0][1]:+.2f} dB)')
print(f'monotone descent: {all(b <= a + 1e-12 for a, b in zip([h[0] for h in hist], [h[0] for h in hist][1:]))}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
it = np.arange(1, len(hist) + 1)

ax1.plot(it, [h[0] for h in hist], color='steelblue')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Waveform MSE')
ax1.set_title('Loss, descending through gla')

ax2.plot(it, [h[1] for h in hist], color='coral')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('SI-SDR (dB)')
ax2.set_title('Reconstruction quality')

plt.tight_layout()
plt.show()

The loss falls monotonically and the reconstruction improves, so the gradient through
`gla` is not just finite — it points somewhere useful. That is the property a training loop
needs.

## 4. A waveform target is the wrong objective downstream of Griffin-Lim

Section 3 works, but read carefully what it optimised: it tuned the magnitudes so that *this
particular Griffin-Lim run* lands nearer the target. That is not the same as making the
magnitudes more correct, and the difference matters as soon as a model is in the loop.

Griffin-Lim from zero phase converges to **a** signal whose magnitudes match — not to the
one you started from. Which signal it picks depends on the initialisation. So the map from
magnitudes to waveform is not monotone in magnitude accuracy, and a waveform loss placed
after it is not measuring what you think.

Two cells demonstrate it. First: hand Griffin-Lim the **exact clean magnitudes** and see
whether the waveform is any closer to the clean signal than the noisy magnitudes were.

In [ ]:
clean_mag = analyse(clean)
noisy_mag = analyse(noisy)

print('Griffin-Lim from zero phase:')
for maxit in (6, 30):
    with torch.no_grad():
        from_clean = float(si_sdr(clean_w, recon(clean_mag, maxit=maxit)))
        from_noisy = float(si_sdr(clean_w, recon(noisy_mag, maxit=maxit)))
    verdict = 'as expected' if from_clean > from_noisy else '<-- INVERTED'
    print(f'  maxit={maxit:3d}:  clean magnitudes {from_clean:7.2f} dB   '
          f'noisy magnitudes {from_noisy:7.2f} dB   {verdict}')

Perfect magnitudes do not reliably beat noisy ones, and the ordering flips with the
iteration count. Nothing is broken — this is what "converges to *a* consistent signal"
means. But it does mean a waveform loss here is largely measuring Griffin-Lim.

Now the fix: **freeze the phase**. Take a PGHI phase once, treat it as a constant, and let
only the magnitudes vary. The map magnitudes → waveform is then linear and deterministic,
so better magnitudes must give a better waveform — and it is still differentiable, as
section 1 measured.

In [ ]:
xp_noisy = torch.zeros(L, dtype=DT)
xp_noisy[:Ls] = torch.as_tensor(noisy, dtype=DT)
with torch.no_grad():
    c_ref, _ = filterbankconstphase(xp_noisy, g, a, L=L, fc=fc, fs=fs)
    fixed_phase = [torch.angle(p) for p in c_ref]


def synth_fixed_phase(mag_flat):
    # Phase is a constant; the gradient flows through the magnitude factor.
    return ifilterbank([m.to(CDT) * torch.exp(1j * p)
                        for m, p in zip(unflatten(mag_flat), fixed_phase)],
                       gd, a, Ls)


with torch.no_grad():
    print('Frozen PGHI phase:')
    fc_ = float(si_sdr(clean_w, synth_fixed_phase(clean_mag)))
    fn_ = float(si_sdr(clean_w, synth_fixed_phase(noisy_mag)))
    print(f'  clean magnitudes {fc_:7.2f} dB   noisy magnitudes {fn_:7.2f} dB')
    print(f'  headroom a magnitude model can exploit: {fc_ - fn_:+.2f} dB')

With the phase frozen the ordering is right: better magnitudes give a better waveform, and
the gap is the headroom a magnitude model could actually win. It is small here because the
frozen phase came from the *noisy* signal and caps the whole path — which is itself the next
thing to improve, and now a measurable one.

The general recipe, in two cells and a few seconds:

1. Reconstruct from the **true** magnitudes. If that is no better than reconstructing from
   the corrupted ones, your waveform loss is measuring the phase step, not your model.
2. If so, make the phase step deterministic (freeze it) or strengthen it (more iterations, a
   better initialisation, a more redundant bank) before building a training loop on it.

## Summary

- Analysis, synthesis and `gla` in `cool_frames.torch` are differentiable;
  `filterbankconstphase` (PGHI) and `ifilterbankiter` are not. Section 1 measures this
  rather than assuming it.
- The `gla` gradient is correct, not merely non-zero (section 2), and descending on it
  reduces the objective monotonically (section 3).
- A correct gradient is not a sufficient reason to use a loss. Downstream of Griffin-Lim
  from zero phase, a waveform target is not monotone in magnitude accuracy; freezing the
  phase restores that and keeps the path differentiable (section 4).

See **Notebook 1** for how PGHI works and **Notebook 2** for a comparison of every
phase-retrieval method the toolbox ships.